In [ ]:
# checking the version of torch
import torch
print(torch.__version__)


2.9.0+cpu


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

ratings = pd.read_csv("ratings.csv")
ratings.head()

user_ids = ratings['userId'].astype("category").cat.codes.values
movie_ids = ratings['movieId'].astype("category").cat.codes.values
ratings_values = ratings['rating'].values

# Creating a custom Dataset
class MovieDataset(Dataset):
    def __init__(self, users, movies, ratings):
        self.users = torch.tensor(users, dtype=torch.long)
        self.movies = torch.tensor(movies, dtype=torch.long)
        self.ratings = torch.tensor(ratings, dtype=torch.float)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

# Creating DataLoader
dataset = MovieDataset(user_ids, movie_ids, ratings_values)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# Defining the recommendation model
class RecommendationModel(nn.Module):
    def __init__(self, num_users, num_movies, embedding_size=50):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.movie_embedding = nn.Embedding(num_movies, embedding_size)

    def forward(self, user, movie):
        user_vec = self.user_embedding(user)
        movie_vec = self.movie_embedding(movie)
        return (user_vec * movie_vec).sum(dim=1)

# Training the model
num_users = len(set(user_ids))
num_movies = len(set(movie_ids))

model = RecommendationModel(num_users, num_movies)


loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for users, movies, ratings in loader:
        optimizer.zero_grad()
        predictions = model(users, movies)
        loss = loss_fn(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 16959.8085
Epoch 2, Loss: 5849.7300
Epoch 3, Loss: 2246.0999
Epoch 4, Loss: 936.6203
Epoch 5, Loss: 526.8965
